# Dimension scaling with exact ground truth

P is a Gaussian mixture, so the projected measure Q\*, the log-partition ψ and the likelihood ratio L\*
are **closed form**: tilting a Gaussian mixture gives another Gaussian mixture. β_true is chosen, the
targets are the exact E_Q\*[x], and the pipeline has to recover β from P_θ samples. Design and
predictions are pre-registered in `taskc/DECISIONS.md` section 25, written before this ran.

Two sweeps over d ∈ {2, 8, 32, 128}: **(a)** a fixed per-coordinate tilt, so KL grows with d; **(b)** β
rescaled so KL(Q\*‖P) = 0.5 nats at every d. One P model per d, shared across both sweeps.

Arms: **SMT**; **weighting + SIR**; an **oracle** diffusion trained on exact Q\* samples (it needs target
samples, and is labelled as such everywhere); and a **SIR-retrain** diffusion.

In [ ]:
PINNED_COMMIT    = "__PINNED__"
NOTEBOOK_VERSION = "dimscale-2026.09.25a"
EXPECT_TASKC     = "taskc-2026.09.25b"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned :", PINNED_COMMIT); print("HEAD   :", HEAD); print("notebook:", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), "checkout missed the pinned commit -- stop"
!git log --oneline -1

### Stage 0 — environment, full fp32 (TF32 off), Drive

In [ ]:
import os, sys, json, math, time, itertools
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
from dataclasses import replace
import taskc
from taskc.config import CFG, FROZEN, frozen
from taskc.data import make_loader
from taskc.ptheta import (make_schedule, build_model, train_ptheta_decay_ema,
                          save_checkpoint, load_checkpoint)
from taskc.sampler import reverse_ancestral
from taskc.hnet import HNet, train_hnet, tower_curve, h0_vs_L
from taskc.smt import make_eps_correction
from taskc.gmm import (GM, make_mixture, gm_sample, tilt, psi, mean_qstar, kl_qstar,
                       log_Lstar, beta_for_kl, heldout_spec, heldout_empirical,
                       solve_beta_samples, resample, sliced_w1)

_nb_path = "notebooks/taskc_10_dimscale_colab.ipynb"
# INVARIANT: PINNED_COMMIT must name a commit whose copy of this notebook already
# carries NOTEBOOK_VERSION -- a version bump takes two commits.
assert taskc.__version__ == EXPECT_TASKC, f"stale: taskc {taskc.__version__} != {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open(_nb_path).read(), (
    f"{NOTEBOOK_VERSION} does not appear in the copy of this notebook at PINNED_COMMIT "
    f"({PINNED_COMMIT[:7]}). Either this is a stale cached notebook -- reopen it from the "
    f"pinned URL -- or the pin was not advanced together with the version.")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_dimscale"
except Exception:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_taskc/dimscale_local")
os.makedirs(DRIVE, exist_ok=True)
print("taskc:", taskc.__version__, "| torch:", torch.__version__, "| device:", DEVICE,
      "| tf32:", torch.backends.cuda.matmul.allow_tf32)
print("out ->", DRIVE)

### Stage 0b — configuration

In [ ]:
DIMS      = [2, 8, 32, 128]        # append 256 only if time allows
B_FIXED   = 0.15                   # sweep (a): per-coordinate tilt (DECISIONS.md 25.2)
KL_FIXED  = 0.5                    # sweep (b): nats
N_TRAIN   = 100_000                # training samples for P_theta, oracle, SIR-retrain
N_POOL    = 500_000                # fresh P_theta draw: dual solve + SIR pool
N_EVAL    = 100_000                # per-arm evaluation sample
N_EXACT   = 200_000                # exact Q* / exact P reference samples
N_HPSI    = 100_000                # h_psi training targets
H_EPOCHS  = 100
N_PROJ    = 8                      # held-out random projections
SKIP_DONE = True

RES = os.path.join(DRIVE, "dimscale.json")
res = json.load(open(RES)) if (SKIP_DONE and os.path.exists(RES)) else dict(
    notebook=NOTEBOOK_VERSION, pinned=PINNED_COMMIT, taskc=taskc.__version__, device=DEVICE,
    device_name=(torch.cuda.get_device_name(0) if DEVICE=="cuda" else DEVICE),
    fp32=dict(tf32=torch.backends.cuda.matmul.allow_tf32),
    dims=DIMS, b_fixed=B_FIXED, kl_fixed=KL_FIXED,
    n_train=N_TRAIN, n_pool=N_POOL, n_eval=N_EVAL, cells={}, base={}, timings={})
def save(): json.dump(res, open(RES,"w"), indent=1, default=float)
save()

def cfg_for(d, **kw):
    """Frozen recipe at dimension d. z_cap is raised because the cap is a path-pathology
    rule tuned to 21-d standardized returns, not to a d-dimensional mixture."""
    return frozen(data_dim=d, artifact_dir=DRIVE, z_cap=1e9, **kw)

def betas_for(gm, d):
    return {"a_fixed_tilt": B_FIXED*np.ones(d),
            "b_fixed_kl":   beta_for_kl(gm, np.ones(d), KL_FIXED)}

print("KL ladder (exact, before any training):")
for d in DIMS:
    gm = make_mixture(d, K=4, seed=0); bb = betas_for(gm, d)
    print(f"  d={d:4d}  (a) KL {kl_qstar(gm,bb['a_fixed_tilt']):7.4f}   "
          f"(b) KL {kl_qstar(gm,bb['b_fixed_kl']):7.4f}  |beta| {np.linalg.norm(bb['b_fixed_kl']):.4f}")
print("done:", list(res["cells"]))

### Stage 1 — one P_θ per d, and the base-model error

The base-model error separates **prior degradation** from **correction error**: if SMT gets worse with d,
part of that may simply be that P_θ itself is a worse fit at larger d. Measured as sliced Wasserstein to
exact P samples, plus coordinate mean and variance errors in SE units.

In [ ]:
def train_model(z, d, tag, seed=0):
    cfg = cfg_for(d)
    m = build_model(cfg).to(DEVICE)
    sch = make_schedule(cfg, device=DEVICE)
    ld = make_loader(z.astype(np.float32), batch_size=cfg.batch_size, seed=seed)
    t0 = time.time()
    m = train_ptheta_decay_ema(m, ld, sch, cfg, device=DEVICE)
    m.eval()
    return m, sch, time.time()-t0

@torch.no_grad()
def draw(model, sched, n, d, seed, eps_correction=None, chunk=50_000):
    cfg = cfg_for(d)
    torch.manual_seed(seed)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(seed)
    out, t0 = [], time.time()
    for lo in range(0, n, chunk):
        m = min(chunk, n-lo)
        x = reverse_ancestral(model, (m, d), sched.alphas, sched.alphas_bar, sched.betas,
                              DEVICE, t_start=sched.t_start, eps_correction=eps_correction)
        out.append(x.detach().cpu().numpy().astype(np.float64))
    return np.concatenate(out, 0), time.time()-t0

for d in DIMS:
    key = f"base_d{d}"
    ck = os.path.join(DRIVE, f"ptheta_d{d}.pt")
    gm = make_mixture(d, K=4, seed=0)
    if key not in res["base"]:
        t0 = time.time()
        zP = gm_sample(gm, N_TRAIN, seed=10_000+d)
        model, sched, tr_s = train_model(zP, d, f"P_d{d}")
        torch.save({"sd": model.state_dict(), "d": d}, ck)
        Xp, samp_s = draw(model, sched, N_EVAL, d, seed=20_000+d)
        Xe = gm_sample(gm, N_EXACT, seed=30_000+d)
        n = Xp.shape[0]
        mu_err = (Xp.mean(0) - gm.pi @ gm.mu) / (Xp.std(0)/math.sqrt(n))
        vt = (gm.pi @ (1.0 + gm.mu**2)) - (gm.pi @ gm.mu)**2
        v_se = np.sqrt(np.maximum(((Xp-Xp.mean(0))**4).mean(0) - Xp.var(0)**2, 0)/n)
        v_err = (Xp.var(0) - vt) / np.where(v_se>0, v_se, np.inf)
        res["base"][key] = dict(d=d, train_seconds=tr_s, sample_seconds=samp_s,
            sw_to_exact_P=float(sliced_w1(Xp, Xe, 256, seed=5)),
            mean_max_z=float(np.abs(mu_err).max()), mean_rms_z=float(np.sqrt((mu_err**2).mean())),
            var_max_z=float(np.abs(v_err).max()), var_rms_z=float(np.sqrt((v_err**2).mean())))
        res["timings"][key] = time.time()-t0; save()
    b = res["base"][key]
    print(f"d={d:4d}  base: SW(P_theta, P) {b['sw_to_exact_P']:.5f}  mean |z| max {b['mean_max_z']:5.2f} "
          f"rms {b['mean_rms_z']:5.2f}  var |z| max {b['var_max_z']:5.2f}  train {b['train_seconds']:.0f}s")

### Stage 2 — the four arms, per d and per sweep

For each cell: recover β from a fresh P_θ pool by the convex dual, train h_ψ on L\* targets, then run
SMT, weighting+SIR, the oracle, and the SIR-retrain. Everything is checked against the **exact** Q\*.

In [ ]:
def se_mean(X):  return X.std(0)/math.sqrt(X.shape[0])

def functional_errors(Xa, spec, exact_var, exact_tail, w=None):
    """Held-out functional errors in SE units, weighted or not."""
    n = Xa.shape[0]
    P = Xa @ spec["U"].T
    if w is None:
        v, t = P.var(0), (P > spec["thresh"][None,:]).mean(0)
        v_se = np.sqrt(np.maximum(((P-P.mean(0))**4).mean(0) - P.var(0)**2, 0)/n)
        t_se = np.sqrt(np.maximum(t*(1-t), 0)/n)
    else:
        w = w/w.sum(); ess = 1.0/np.sum(w**2)
        m = w@P; v = w@(P**2) - m**2
        ind = (P > spec["thresh"][None,:]).astype(float); t = w@ind
        v_se = np.sqrt(np.maximum(w @ ((P-m)**4) - v**2, 0)/ess)   # w is (n,), (P-m)**4 is (n, n_proj)
        t_se = np.sqrt(np.maximum(t*(1-t), 0)/ess)
    ez = (v-exact_var)/np.where(v_se>0, v_se, np.inf)
    tz = (t-exact_tail)/np.where(t_se>0, t_se, np.inf)
    return ez, tz

for d in DIMS:
    gm = make_mixture(d, K=4, seed=0)
    cfg = cfg_for(d)
    sched = make_schedule(cfg, device=DEVICE)
    model = build_model(cfg).to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(DRIVE, f"ptheta_d{d}.pt"),
                                     map_location=DEVICE)["sd"]); model.eval()
    for sweep, beta_true in betas_for(gm, d).items():
        key = f"d{d}_{sweep}"
        if key in res["cells"]:
            print(f"{key}: cached"); continue
        t_cell = time.time()
        q_exact = tilt(gm, beta_true)
        c = mean_qstar(gm, beta_true)                      # exact targets
        spec = heldout_spec(q_exact, n_proj=N_PROJ, seed=77)
        Xq_exact = gm_sample(q_exact, N_EXACT, seed=40_000+d)

        # --- recover beta from P_theta samples ---------------------------------
        Xpool, pool_s = draw(model, sched, N_POOL, d, seed=50_000+d)
        beta_hat, w_pool, kl_hat, ess = solve_beta_samples(Xpool, c)
        # Did the dual actually move? A prior whose support does not cover the targets
        # leaves beta_hat at ~0 with the constraints unmet, which otherwise looks like a
        # clean solve. Report the residual in units of the pool's own scale.
        resid = float(np.max(np.abs(w_pool @ Xpool - c) / np.maximum(Xpool.std(0), 1e-12)))
        if resid > 1e-3:
            print(f"  WARNING {key}: dual residual {resid:.3e} (sd units) -- constraints not met; "
                  f"|beta_hat| {np.linalg.norm(beta_hat):.4e}. Treat this cell as a solve failure.")
        L_pool = np.exp(Xpool @ beta_hat - (np.log(np.mean(np.exp(Xpool@beta_hat - (Xpool@beta_hat).max())))
                                            + (Xpool@beta_hat).max()))
        arms, tim = {}, {}

        # --- arm 2: weighting + SIR -------------------------------------------
        t0 = time.time()
        X_sir = resample(Xpool, w_pool, N_EVAL, seed=60_000+d)
        tim["weight_sir"] = time.time()-t0 + pool_s
        arms["weight_sir"] = dict(X=X_sir, w_on=Xpool, w=w_pool)

        # --- arm 1: SMT --------------------------------------------------------
        t0 = time.time()
        sub = np.random.default_rng(70_000+d).choice(len(Xpool), size=min(N_HPSI, len(Xpool)), replace=False)
        # h_min must sit below the smallest L*, which at large KL is far under 0.05
        h_min = float(max(1e-4, 0.5*L_pool[sub].min()))
        hn = HNet(d, 256, 32, h_min)
        train_hnet(hn, Xpool[sub], L_pool[sub], sched, device=DEVICE, epochs=H_EPOCHS, verbose=False)
        h_train_s = time.time()-t0
        X_smt, smt_s = draw(model, sched, N_EVAL, d, seed=80_000+d,
                            eps_correction=make_eps_correction(hn, sched))
        tim["SMT"] = h_train_s + smt_s
        arms["SMT"] = dict(X=X_smt)
        tow = tower_curve(hn, Xpool[sub][:20_000], sched, (0, 100, 300, 600, 900), device=DEVICE)
        h0L = h0_vs_L(hn, Xpool[sub][:20_000], L_pool[sub][:20_000], sched, device=DEVICE)

        # --- arm 3: oracle on exact Q* samples --------------------------------
        zo = gm_sample(q_exact, N_TRAIN, seed=90_000+d)
        mo, _, o_train = train_model(zo, d, f"oracle_{key}")
        X_or, o_samp = draw(mo, sched, N_EVAL, d, seed=100_000+d)
        tim["oracle"] = o_train + o_samp
        arms["oracle"] = dict(X=X_or)

        # --- arm 4: retrain on SIR output -------------------------------------
        zr = resample(Xpool, w_pool, N_TRAIN, seed=110_000+d)
        mr, _, r_train = train_model(zr, d, f"sirretrain_{key}")
        X_rt, r_samp = draw(mr, sched, N_EVAL, d, seed=120_000+d)
        tim["sir_retrain"] = r_train + r_samp
        arms["sir_retrain"] = dict(X=X_rt)

        # --- metrics -----------------------------------------------------------
        cell = dict(d=d, sweep=sweep, kl_exact=float(kl_qstar(gm, beta_true)), kl_hat=float(kl_hat),
                    ess=float(ess), beta_err=float(np.abs(beta_hat-beta_true).max()),
                    beta_rel=float(np.linalg.norm(beta_hat-beta_true)/np.linalg.norm(beta_true)),
                    dual_residual_sd=resid, beta_hat_norm=float(np.linalg.norm(beta_hat)),
                    tower={str(t): v["mean"] for t, v in tow.items()},
                    tower_full=tow, h_min=h_min, h0_vs_L=h0L, seconds=tim, arms={})
        for name, a in arms.items():
            X = a["X"]
            if name == "weight_sir":      # functionals from the weighted estimator
                W = a["w"]; XW = a["w_on"]
                mz = (W@XW - c)/np.sqrt(np.maximum(W@(XW**2)-(W@XW)**2, 0)/(1.0/np.sum(W**2)))
                ez, tz = functional_errors(XW, spec, spec["var"], spec["tail"], w=W)
            else:
                mz = (X.mean(0) - c)/np.where(se_mean(X)>0, se_mean(X), np.inf)
                ez, tz = functional_errors(X, spec, spec["var"], spec["tail"])
            cell["arms"][name] = dict(
                mean_max_z=float(np.abs(mz).max()), mean_rms_z=float(np.sqrt((mz**2).mean())),
                var_max_z=float(np.abs(ez).max()), tail_max_z=float(np.abs(tz).max()),
                sw_to_exact_Q=float(sliced_w1(X, Xq_exact, 256, seed=9)),
                unique_frac=float(len(np.unique(X, axis=0))/len(X)),
                seconds=tim[name])
        res["cells"][key] = cell; res["timings"][key] = time.time()-t_cell; save()
        print(f"{key}: KL {cell['kl_exact']:.3f} ESS {ess*100:6.2f}%  beta err {cell['beta_err']:.4f}  "
              f"corr(h0,L*) {h0L.get('corr', float('nan')):.4f}", flush=True)
        for n_, v in cell["arms"].items():
            print(f"    {n_:12s} mean|z| {v['mean_max_z']:7.2f}/{v['mean_rms_z']:5.2f}  var|z| {v['var_max_z']:7.2f}  "
                  f"tail|z| {v['tail_max_z']:7.2f}  SW {v['sw_to_exact_Q']:.5f}  uniq {v['unique_frac']*100:5.1f}%  "
                  f"{v['seconds']:6.0f}s", flush=True)

### Stage 3 — tables

The crossover in prediction 1 is read off the sweep-(a) table: the dimension at which SMT's
error drops below weighting's.

In [ ]:
ARMS = ["SMT","weight_sir","oracle","sir_retrain"]
LAB  = {"SMT":"SMT","weight_sir":"weight+SIR","oracle":"oracle [TARGET SAMPLES]","sir_retrain":"SIR-retrain"}
for sweep, title in (("a_fixed_tilt","(a) fixed per-coordinate tilt -- KL grows with d"),
                     ("b_fixed_kl", f"(b) fixed KL = {KL_FIXED} nats")):
    print(f"\n{'='*118}\n{title}\n{'='*118}")
    print(f"{'d':>5s} {'KL':>7s} {'ESS %':>7s} {'beta err':>9s}  " +
          "".join(f"{LAB[a]:>26s}" for a in ARMS))
    print(f"{'':>5s} {'':>7s} {'':>7s} {'':>9s}  " + "".join(f"{'mean|z| / SW':>26s}" for a in ARMS))
    for d in DIMS:
        k = f"d{d}_{sweep}"
        if k not in res["cells"]: continue
        c = res["cells"][k]
        row = f"{d:5d} {c['kl_exact']:7.3f} {c['ess']*100:7.2f} {c['beta_err']:9.4f}  "
        for a in ARMS:
            v = c["arms"][a]
            row += f"{v['mean_max_z']:11.2f} /{v['sw_to_exact_Q']:12.5f}"
        print(row)
    print(f"\n{'d':>5s} " + "".join(f"{LAB[a]+' uniq/tail|z|':>28s}" for a in ARMS))
    for d in DIMS:
        k = f"d{d}_{sweep}"
        if k not in res["cells"]: continue
        c = res["cells"][k]
        print(f"{d:5d} " + "".join(f"{c['arms'][a]['unique_frac']*100:16.1f}%/{c['arms'][a]['tail_max_z']:9.2f}"
                                   for a in ARMS))

print(f"\n{'='*70}\nbase-model error (P_theta vs exact P) -- prior degradation, not correction error\n{'='*70}")
print(f"{'d':>5s} {'SW(P_theta,P)':>15s} {'mean |z| max':>14s} {'var |z| max':>13s} {'train s':>9s}")
for d in DIMS:
    b = res["base"].get(f"base_d{d}")
    if b: print(f"{d:5d} {b['sw_to_exact_P']:15.5f} {b['mean_max_z']:14.2f} {b['var_max_z']:13.2f} {b['train_seconds']:9.0f}")

print(f"\n{'='*70}\nh-diagnostics\n{'='*70}")
print(f"{'cell':>22s} {'corr(h0,L*)':>12s}  tower property")
for k, c in res["cells"].items():
    tw = " ".join(f"t={t}:{v:.3f}" for t, v in c["tower"].items())
    print(f"{k:>22s} {c['h0_vs_L'].get('corr', float('nan')):12.4f}  {tw}   (target 1.000)")
print("\ntimings (min):", {k: round(v/60,2) for k,v in res["timings"].items()})

Results in `dimscale.json` on Drive. Predictions in DECISIONS.md section 25.5 are checked against
these tables and any that fail are recorded there as falsified.